# OpenMantra Pipeline Evaluation

Evaluates the full manga translation pipeline on OpenMantra dataset:
- **OCR Accuracy**: CER on matched bubbles
- **Translation Quality**: BLEU, chrF++, BERTScore, COMET on matched bubbles  
- **Reading Order**: Kendall's Tau correlation
- **Coverage**: GT text coverage, bubble utilization

Uses containment-based matching: GT text bbox must be ≥80% inside predicted bubble.


In [ ]:
import os
import sys

ENDSWITH = 'OpenMantra'
NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDSWITH):
    raise ValueError(f"Expected dir ending with {ENDSWITH}, got {NOTEBOOK_DIR}")

BASE_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', '..', '..', '..')
sys.path.insert(0, BASE_DIR)
print(f"Base dir: {BASE_DIR}")


In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from typing import List, Dict, Tuple, Any, Optional, Union
from tqdm.auto import tqdm
from scipy.stats import kendalltau
import torch
from torch.utils.data import Dataset
from torchmetrics.text import CharErrorRate, WordErrorRate, BLEUScore, SacreBLEUScore, CHRFScore
from torchmetrics.text.bert import BERTScore

try:
    from comet import download_model, load_from_checkpoint
    COMET_AVAILABLE = True
except ImportError:
    COMET_AVAILABLE = False
    print("Warning: COMET not available. Install with: pip install unbabel-comet")

from src.pipeline.SegmentationModels.YoloSeg import YoloBubbleSeg, YoloPanelSeg
from src.pipeline.SegmentationModels.BubbleSegmenterWithSplit import BubbleSegmenterWithSplit
from src.pipeline.SegmentationModels.BubbleSegmentationWithOrdering import BubbleSegmentationWithOrdering
from src.pipeline.OCRModels.MangaOCRModel import MangaOCRModel
from src.pipeline.TranslationModels.ElanMtJaEnTranslator import ElanMtJaEnTranslator
from src.pipeline.TranslationModels.ContextAwareLLMTranslator import ContextAwareLLMTranslator
from src.pipeline.Utils.MangaTypesetter import MangaTypesetter
from src.pipeline.Utils.MangaPipeline import MangaPipeline

print("Imports successful!")


In [ ]:
class OpenMantraDataset(Dataset):
    """Dataset class for OpenMantra manga translation dataset."""
    
    def __init__(
        self,
        root_dir: str,
        annotation_file: str = "annotation.json",
        transform: Optional[Any] = None,
        language: str = "ja",
        book_titles: Optional[List[str]] = None
    ):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.language = language
        
        annotation_path = self.root_dir / annotation_file
        with open(annotation_path, 'r', encoding='utf-8') as f:
            self.annotations = json.load(f)
        
        if book_titles is not None:
            self.annotations = [
                book for book in self.annotations 
                if book['book_title'] in book_titles
            ]
        
        self.samples: List[Tuple[Dict, Dict]] = []
        for book in self.annotations:
            book_title = book['book_title']
            for page in book['pages']:
                self.samples.append(({'book_title': book_title}, page))
    
    def __len__(self) -> int:
        return len(self.samples)
    
    def __getitem__(self, idx: int) -> Tuple[Image.Image, Dict, List[Dict]]:
        book_info, page_info = self.samples[idx]
        
        image_paths = page_info.get('image_paths', {})
        relative_image_path = image_paths.get(self.language, '')
        image_path = self.root_dir / relative_image_path
        
        image = Image.open(image_path).convert('RGB')
        image_size = image.size
        
        image_info = {
            'book_title': book_info['book_title'],
            'page_index': page_info.get('page_index', -1),
            'image_path': str(image_path),
            'image_size': image_size,
            'frames': page_info.get('frame', [])
        }
        
        raw_bubbles = page_info.get('text', [])
        bubbles = []
        for bubble in raw_bubbles:
            converted_bubble = {
                'xmin': bubble['x'],
                'ymin': bubble['y'],
                'xmax': bubble['x'] + bubble['w'],
                'ymax': bubble['y'] + bubble['h'],
                'text_ja': bubble.get('text_ja', ''),
                'text_en': bubble.get('text_en', ''),
                'text_zh': bubble.get('text_zh', '')
            }
            bubbles.append(converted_bubble)
        
        if self.transform is not None:
            image = self.transform(image)
        
        return image, image_info, bubbles
    
    def get_book_titles(self) -> List[str]:
        return list(set(book['book_title'] for book in self.annotations))


In [ ]:
# Configuration
DEVICE = "mps"  # Change to "cuda" or "cpu" as needed
OPENMANTRA_ROOT = os.path.join(BASE_DIR, "data/open-mantra-dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "output/pipeline/Evaluators/Pipeline/OpenMantra")
CONTAINMENT_THRESHOLD = 0.8  # GT text must be 80% inside bubble

from datetime import datetime
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(OUTPUT_DIR, f"run_{RUN_ID}")

PLOTS_DIR = os.path.join(RUN_DIR, "plots")
VIS_DIR = os.path.join(RUN_DIR, "visualizations")
TRANS_DIR = os.path.join(RUN_DIR, "translations")

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)
os.makedirs(TRANS_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"OpenMantra root: {OPENMANTRA_ROOT}")
print(f"Run dir: {RUN_DIR}")


## 1. Load OpenMantra Dataset


In [ ]:
dataset = OpenMantraDataset(root_dir=OPENMANTRA_ROOT)

print(f"Total pages: {len(dataset)}")
print(f"Books: {dataset.get_book_titles()}")


## 2. Create Pipeline


In [ ]:
panel_detector = YoloPanelSeg(variant="v8n", device=DEVICE, verbose=False)

bubble_detector = BubbleSegmenterWithSplit(
    variant="v8s",
    device=DEVICE,
    verbose=False,
    plot=False
)

ordered_segmenter = BubbleSegmentationWithOrdering(
    bubble_detector=bubble_detector,
    panel_detector=panel_detector,
    num_panel_rows=4,
    right_to_left=True,
    verbose=False,
    plot=False
)

ocr_model = MangaOCRModel(verbose=False)
# translator = ElanMtJaEnTranslator(elan_model="tiny", device=DEVICE, verbose=False)
translator = ContextAwareLLMTranslator(model_name='TheBlindMaster/Qwen2.5-0.5B-Instruct-emergent-finetune-train_full_context-all-full-r32', context_window=3, verbose=True)

typesetter = MangaTypesetter()

pipeline = MangaPipeline(
    segmenter=ordered_segmenter,
    ocr_model=ocr_model,
    translator=translator,
    typesetter=typesetter,
    verbose=False
)

print("Pipeline created!")


## 3. Evaluation Functions


In [ ]:
def match_bubbles_to_gt_texts(
    pred_bboxes: List[List[float]], 
    gt_texts: List[Dict],
    containment_threshold: float = 0.8
) -> Tuple[Dict[int, List[int]], List[int], List[int]]:
    """
    Match GT text bboxes to predicted bubbles based on containment.
    
    Args:
        pred_bboxes: List of predicted bubble bboxes [x1, y1, x2, y2]
        gt_texts: List of GT text dicts with 'xmin', 'ymin', 'xmax', 'ymax', 'text_ja', 'text_en'
        containment_threshold: Fraction of GT text that must be inside bubble
    
    Returns:
        matches: Dict mapping pred_idx -> [list of gt_indices it contains]
        unmatched_gt: GT text indices not contained in any bubble
        empty_bubbles: Predicted bubble indices containing no GT text
    """
    matches = {i: [] for i in range(len(pred_bboxes))}
    matched_gt = set()
    
    for pred_idx, bbox in enumerate(pred_bboxes):
        bx1, by1, bx2, by2 = bbox
        
        for gt_idx, gt in enumerate(gt_texts):
            gx1, gy1 = gt['xmin'], gt['ymin']
            gx2, gy2 = gt['xmax'], gt['ymax']
            
            # Compute intersection
            ix1 = max(bx1, gx1)
            iy1 = max(by1, gy1)
            ix2 = min(bx2, gx2)
            iy2 = min(by2, gy2)
            
            if ix1 < ix2 and iy1 < iy2:
                intersection = (ix2 - ix1) * (iy2 - iy1)
                gt_area = (gx2 - gx1) * (gy2 - gy1)
                
                if gt_area > 0:
                    containment = intersection / gt_area
                    if containment >= containment_threshold:
                        matches[pred_idx].append(gt_idx)
                        matched_gt.add(gt_idx)
    
    unmatched_gt = [i for i in range(len(gt_texts)) if i not in matched_gt]
    empty_bubbles = [i for i, texts in matches.items() if len(texts) == 0]
    
    return matches, unmatched_gt, empty_bubbles


In [ ]:
def evaluate_ocr_page(
    ocr_texts: List[str],
    gt_texts: List[Dict],
    matches: Dict[int, List[int]]
) -> Tuple[List[str], List[str]]:
    """
    Prepare OCR predictions and expected texts for a page.
    Concatenates GT texts for bubbles containing multiple text regions.
    
    Returns:
        (predictions, expected) lists for this page
    """
    predictions = []
    expected = []
    
    for pred_idx, gt_indices in matches.items():
        if not gt_indices:
            continue
        
        # Sort GT indices to maintain reading order, then concatenate
        sorted_gt_indices = sorted(gt_indices)
        expected_text = ''.join([gt_texts[i]['text_ja'] for i in sorted_gt_indices])
        ocr_output = ocr_texts[pred_idx] if pred_idx < len(ocr_texts) else ''
        
        predictions.append(ocr_output)
        expected.append(expected_text)
    
    return predictions, expected


def evaluate_translation_page(
    translations: List[str],
    ocr_texts: List[str],
    gt_texts: List[Dict],
    matches: Dict[int, List[int]]
) -> Tuple[List[str], List[str], List[str]]:
    """
    Prepare translation predictions, expected texts, and source texts for a page.
    Concatenates GT translations for bubbles containing multiple text regions.
    
    Returns:
        (predictions, expected, sources) lists for this page
    """
    predictions = []
    expected = []
    sources = []
    
    for pred_idx, gt_indices in matches.items():
        if not gt_indices:
            continue
        
        sorted_gt_indices = sorted(gt_indices)
        expected_trans = ' '.join([gt_texts[i]['text_en'] for i in sorted_gt_indices])
        pred_trans = translations[pred_idx] if pred_idx < len(translations) else ''
        source_text = ocr_texts[pred_idx] if pred_idx < len(ocr_texts) else ''
        
        predictions.append(pred_trans)
        expected.append(expected_trans)
        sources.append(source_text)
    
    return predictions, expected, sources


In [ ]:
def evaluate_ordering_page(
    matches: Dict[int, List[int]],
    num_gt_texts: int
) -> Dict[str, Any]:
    """
    Evaluate reading order for a page.
    
    Compares the order of predicted bubbles (by their index) 
    against GT reading order (GT indices are already in reading order).
    
    Returns:
        Dict with kendall_tau, exact_match, num_matched_bubbles
    """
    # Build mapping: for bubbles with GT, what's the minimum GT index they contain?
    # This represents the "reading position" of each bubble
    bubble_first_gt = {}
    for pred_idx, gt_indices in matches.items():
        if gt_indices:
            bubble_first_gt[pred_idx] = min(gt_indices)
    
    if len(bubble_first_gt) < 2:
        return {
            'kendall_tau': 1.0,
            'exact_match': True,
            'num_matched_bubbles': len(bubble_first_gt)
        }
    
    # Get prediction order (pred indices in their natural order)
    pred_order = sorted(bubble_first_gt.keys())
    
    # Get the GT ranks for each predicted bubble (in prediction order)
    gt_ranks = [bubble_first_gt[p] for p in pred_order]
    
    # Expected: gt_ranks should be monotonically increasing
    # Compute Kendall's Tau: correlation between [0,1,2,...] and gt_ranks
    ideal_sequence = list(range(len(gt_ranks)))
    tau, _ = kendalltau(ideal_sequence, gt_ranks)
    
    # Handle NaN (can happen if all values are identical)
    if np.isnan(tau):
        tau = 1.0
    
    # Check exact match (is gt_ranks sorted?)
    is_sorted = all(gt_ranks[i] <= gt_ranks[i+1] for i in range(len(gt_ranks)-1))
    
    return {
        'kendall_tau': tau,
        'exact_match': is_sorted,
        'num_matched_bubbles': len(bubble_first_gt)
    }


In [ ]:
def evaluate_page(
    pipeline: MangaPipeline,
    image: np.ndarray,
    gt_texts: List[Dict],
    containment_threshold: float = 0.8,
    conf_threshold: float = 0.5
) -> Dict[str, Any]:
    """
    Evaluate pipeline on a single page.
    
    Returns dict with:
        - ocr_predictions, ocr_expected: Lists for OCR evaluation
        - trans_predictions, trans_expected: Lists for translation evaluation
        - ordering: Dict with ordering metrics
        - coverage: Dict with coverage metrics
    """
    # Run pipeline
    _, results = pipeline.process(image, conf_threshold=conf_threshold, return_intermediate=True)
    
    pred_bboxes = results.get('bboxes', [])
    ocr_texts = results.get('ocr_texts', [])
    translations = results.get('translated_texts', [])
    
    # Filter out empty GT texts
    valid_gt_texts = [gt for gt in gt_texts if gt.get('text_ja', '').strip() and gt.get('text_en', '').strip()]
    
    if len(pred_bboxes) == 0 or len(valid_gt_texts) == 0:
        return {
            'ocr_predictions': [],
            'ocr_expected': [],
            'trans_predictions': [],
            'trans_expected': [],
            'trans_sources': [],
            'ordering': {'kendall_tau': 0.0, 'exact_match': False, 'num_matched_bubbles': 0},
            'coverage': {
                'gt_coverage': 0.0,
                'bubble_utilization': 0.0,
                'num_gt_texts': len(valid_gt_texts),
                'num_pred_bubbles': len(pred_bboxes),
                'num_matched_gt': 0,
                'num_unmatched_gt': len(valid_gt_texts),
                'num_empty_bubbles': len(pred_bboxes)
            },
            'pred_bboxes': pred_bboxes,
            'valid_gt_texts': valid_gt_texts,
            'matches': {}
        }
    
    # Match bubbles to GT texts
    matches, unmatched_gt, empty_bubbles = match_bubbles_to_gt_texts(
        pred_bboxes, valid_gt_texts, containment_threshold
    )
    
    # Evaluate OCR
    ocr_preds, ocr_expected = evaluate_ocr_page(ocr_texts, valid_gt_texts, matches)
    
    # Evaluate Translation
    trans_preds, trans_expected, trans_sources = evaluate_translation_page(
        translations, ocr_texts, valid_gt_texts, matches
    )
    
    # Evaluate Ordering
    ordering = evaluate_ordering_page(matches, len(valid_gt_texts))
    
    # Coverage metrics
    num_matched_gt = len(valid_gt_texts) - len(unmatched_gt)
    coverage = {
        'gt_coverage': num_matched_gt / len(valid_gt_texts) if valid_gt_texts else 1.0,
        'bubble_utilization': (len(pred_bboxes) - len(empty_bubbles)) / len(pred_bboxes) if pred_bboxes else 1.0,
        'num_gt_texts': len(valid_gt_texts),
        'num_pred_bubbles': len(pred_bboxes),
        'num_matched_gt': num_matched_gt,
        'num_unmatched_gt': len(unmatched_gt),
        'num_empty_bubbles': len(empty_bubbles)
    }
    
    return {
        'ocr_predictions': ocr_preds,
        'ocr_expected': ocr_expected,
        'trans_predictions': trans_preds,
        'trans_expected': trans_expected,
        'trans_sources': trans_sources,
        'ordering': ordering,
        'coverage': coverage,
        'pred_bboxes': pred_bboxes,
        'valid_gt_texts': valid_gt_texts,
        'matches': matches
    }


In [ ]:
def compute_final_metrics(all_results: List[Dict], device: str = 'cpu') -> Dict[str, float]:
    """
    Aggregate results from all pages into final metrics.
    """
    # Collect all predictions/expected across pages
    all_ocr_pred = []
    all_ocr_expected = []
    all_trans_pred = []
    all_trans_expected = []
    all_trans_sources = []
    
    ordering_taus = []
    ordering_exact = []
    coverages = []
    utilizations = []
    
    for result in all_results:
        all_ocr_pred.extend(result['ocr_predictions'])
        all_ocr_expected.extend(result['ocr_expected'])
        all_trans_pred.extend(result['trans_predictions'])
        all_trans_expected.extend(result['trans_expected'])
        all_trans_sources.extend(result.get('trans_sources', []))
        
        if result['ordering']['num_matched_bubbles'] >= 2:
            ordering_taus.append(result['ordering']['kendall_tau'])
            ordering_exact.append(result['ordering']['exact_match'])
        
        coverages.append(result['coverage']['gt_coverage'])
        utilizations.append(result['coverage']['bubble_utilization'])
    
    metrics = {}
    
    # OCR Metrics
    if all_ocr_pred and all_ocr_expected:
        cer_metric = CharErrorRate()
        metrics['ocr_cer'] = cer_metric(all_ocr_pred, all_ocr_expected).item()
    else:
        metrics['ocr_cer'] = 1.0
    
    # Translation Metrics
    if all_trans_pred and all_trans_expected:
        cer_metric = CharErrorRate()
        wer_metric = WordErrorRate()
        bleu_metric = BLEUScore()
        chrf_metric = CHRFScore(n_char_order=6, n_word_order=2)
        
        bleu_refs = [[ref] for ref in all_trans_expected]
        
        metrics['trans_cer'] = cer_metric(all_trans_pred, all_trans_expected).item()
        metrics['trans_wer'] = wer_metric(all_trans_pred, all_trans_expected).item()
        metrics['trans_bleu'] = bleu_metric(all_trans_pred, bleu_refs).item()
        metrics['trans_chrf_pp'] = chrf_metric(all_trans_pred, bleu_refs).item()
        
        # BERTScore
        try:
            bertscore_metric = BERTScore()
            bertscore_result = bertscore_metric(all_trans_pred, all_trans_expected)
            
            # Handle different return types (tensor or list)
            f1_vals = bertscore_result['f1']
            p_vals = bertscore_result['precision']
            r_vals = bertscore_result['recall']
            
            if isinstance(f1_vals, torch.Tensor):
                metrics['trans_bertscore_f1'] = f1_vals.mean().item()
                metrics['trans_bertscore_precision'] = p_vals.mean().item()
                metrics['trans_bertscore_recall'] = r_vals.mean().item()
            else:
                # If it's a list or numpy array, convert to numpy and compute mean
                metrics['trans_bertscore_f1'] = float(np.mean(f1_vals))
                metrics['trans_bertscore_precision'] = float(np.mean(p_vals))
                metrics['trans_bertscore_recall'] = float(np.mean(r_vals))
        except Exception as e:
            print(f"Warning: BERTScore computation failed: {e}")
            metrics['trans_bertscore_f1'] = 0.0
            metrics['trans_bertscore_precision'] = 0.0
            metrics['trans_bertscore_recall'] = 0.0
        
        # COMET
        if COMET_AVAILABLE and all_trans_sources:
            try:
                print("Loading COMET model...")
                model_path = download_model("Unbabel/wmt22-comet-da")
                comet_model = load_from_checkpoint(model_path)
                
                print("Computing COMET scores...")
                comet_data = [
                    {
                        "src": src,
                        "mt": pred,
                        "ref": ref
                    }
                    for src, pred, ref in zip(all_trans_sources, all_trans_pred, all_trans_expected)
                    if src.strip() and pred.strip() and ref.strip()
                ]
                
                if comet_data:
                    comet_scores = comet_model.predict(
                        comet_data,
                        batch_size=8,
                        gpus=1 if device in ['cuda', 'mps'] else 0
                    )
                    metrics['trans_comet'] = np.mean(comet_scores.scores)
                else:
                    metrics['trans_comet'] = 0.0
                    
                del comet_model
                torch.cuda.empty_cache() if torch.cuda.is_available() else None
            except Exception as e:
                print(f"Warning: COMET computation failed: {e}")
                metrics['trans_comet'] = 0.0
        else:
            if not COMET_AVAILABLE:
                print("Warning: COMET not available, skipping COMET metric")
            elif not all_trans_sources:
                print("Warning: No source texts available for COMET")
            metrics['trans_comet'] = 0.0
    else:
        metrics['trans_cer'] = 1.0
        metrics['trans_wer'] = 1.0
        metrics['trans_bleu'] = 0.0
        metrics['trans_chrf_pp'] = 0.0
        metrics['trans_bertscore_f1'] = 0.0
        metrics['trans_bertscore_precision'] = 0.0
        metrics['trans_bertscore_recall'] = 0.0
        metrics['trans_comet'] = 0.0
    
    # Ordering Metrics
    metrics['ordering_kendall_tau'] = np.mean(ordering_taus) if ordering_taus else 0.0
    metrics['ordering_exact_match_rate'] = np.mean(ordering_exact) if ordering_exact else 0.0
    
    # Coverage Metrics
    metrics['gt_coverage_mean'] = np.mean(coverages) if coverages else 0.0
    metrics['bubble_utilization_mean'] = np.mean(utilizations) if utilizations else 0.0
    
    # Counts
    metrics['num_pages'] = len(all_results)
    metrics['num_ocr_samples'] = len(all_ocr_pred)
    metrics['num_trans_samples'] = len(all_trans_pred)
    
    return metrics


In [ ]:
def visualize_matching(
    image: np.ndarray,
    pred_bboxes: List[List[float]],
    gt_texts: List[Dict],
    matches: Dict[int, List[int]],
    save_path: Optional[str] = None
):
    """Visualize GT vs Predicted bboxes with matching."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    
    # Plot 1: GT text bboxes
    axes[0].imshow(image)
    axes[0].set_title("GT Text Bboxes", fontsize=12)
    for i, gt in enumerate(gt_texts):
        rect = plt.Rectangle(
            (gt['xmin'], gt['ymin']), 
            gt['xmax'] - gt['xmin'], 
            gt['ymax'] - gt['ymin'],
            fill=False, edgecolor='green', linewidth=2
        )
        axes[0].add_patch(rect)
        axes[0].text(gt['xmin'], gt['ymin'] - 5, f"GT{i}", fontsize=8, color='green')
    axes[0].axis('off')
    
    # Plot 2: Predicted bubbles with order
    axes[1].imshow(image)
    axes[1].set_title("Predicted Bubbles (Ordered)", fontsize=12)
    colors = plt.cm.rainbow(np.linspace(0, 1, len(pred_bboxes)))
    for i, bbox in enumerate(pred_bboxes):
        rect = plt.Rectangle(
            (bbox[0], bbox[1]), 
            bbox[2] - bbox[0], 
            bbox[3] - bbox[1],
            fill=False, edgecolor=colors[i], linewidth=2
        )
        axes[1].add_patch(rect)
        axes[1].text(bbox[0], bbox[1] - 5, f"[{i}]", fontsize=10, color=colors[i], fontweight='bold')
    axes[1].axis('off')
    
    # Plot 3: Matching overlay
    axes[2].imshow(image)
    axes[2].set_title("Matching (Green=GT, Colors=Pred)", fontsize=12)
    for i, gt in enumerate(gt_texts):
        rect = plt.Rectangle(
            (gt['xmin'], gt['ymin']), 
            gt['xmax'] - gt['xmin'], 
            gt['ymax'] - gt['ymin'],
            fill=False, edgecolor='green', linewidth=1, linestyle='--'
        )
        axes[2].add_patch(rect)
    for pred_idx, gt_indices in matches.items():
        if pred_idx < len(pred_bboxes):
            bbox = pred_bboxes[pred_idx]
            color = 'blue' if gt_indices else 'red'
            rect = plt.Rectangle(
                (bbox[0], bbox[1]), 
                bbox[2] - bbox[0], 
                bbox[3] - bbox[1],
                fill=False, edgecolor=color, linewidth=2
            )
            axes[2].add_patch(rect)
            label = f"P{pred_idx}→GT{gt_indices}" if gt_indices else f"P{pred_idx}(empty)"
            axes[2].text(bbox[0], bbox[1] - 5, label, fontsize=7, color=color)
    axes[2].axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()


def save_translation_output(
    ocr_preds: List[str],
    ocr_expected: List[str],
    trans_preds: List[str],
    trans_expected: List[str],
    trans_sources: List[str],
    image_info: Dict,
    save_path: str
):
    """Save OCR and translation outputs for a page."""
    output = {
        'image_info': image_info,
        'results': []
    }
    
    max_len = max(len(ocr_preds), len(trans_preds), len(trans_sources))
    for i in range(max_len):
        output['results'].append({
            'bubble_idx': i,
            'ocr_predicted': ocr_preds[i] if i < len(ocr_preds) else '',
            'ocr_expected': ocr_expected[i] if i < len(ocr_expected) else '',
            'translation_source': trans_sources[i] if i < len(trans_sources) else '',
            'translation_predicted': trans_preds[i] if i < len(trans_preds) else '',
            'translation_expected': trans_expected[i] if i < len(trans_expected) else ''
        })
    
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)


def create_summary_plots(all_results: List[Dict], save_dir: str):
    """Create summary plots for the evaluation."""
    # Plot 1: Coverage distribution
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    coverages = [r['coverage']['gt_coverage'] for r in all_results]
    axes[0, 0].hist(coverages, bins=20, edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(np.mean(coverages), color='red', linestyle='--', label=f'Mean: {np.mean(coverages):.3f}')
    axes[0, 0].set_xlabel('GT Coverage')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('GT Coverage Distribution')
    axes[0, 0].legend()
    
    # Plot 2: Kendall's Tau distribution
    taus = [r['ordering']['kendall_tau'] for r in all_results if r['ordering']['num_matched_bubbles'] >= 2]
    axes[0, 1].hist(taus, bins=20, edgecolor='black', alpha=0.7, color='orange')
    axes[0, 1].axvline(np.mean(taus), color='red', linestyle='--', label=f'Mean: {np.mean(taus):.3f}')
    axes[0, 1].set_xlabel("Kendall's Tau")
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Ordering Accuracy Distribution')
    axes[0, 1].legend()
    
    # Plot 3: Bubble counts
    gt_counts = [r['coverage']['num_gt_texts'] for r in all_results]
    pred_counts = [r['coverage']['num_pred_bubbles'] for r in all_results]
    x = range(len(all_results))
    axes[1, 0].scatter(gt_counts, pred_counts, alpha=0.5)
    axes[1, 0].plot([0, max(gt_counts)], [0, max(gt_counts)], 'r--', label='Perfect match')
    axes[1, 0].set_xlabel('GT Text Count')
    axes[1, 0].set_ylabel('Predicted Bubble Count')
    axes[1, 0].set_title('GT vs Predicted Counts')
    axes[1, 0].legend()
    
    # Plot 4: Per-book metrics
    from collections import defaultdict
    book_metrics = defaultdict(list)
    for r in all_results:
        book = r['image_info']['book_title']
        book_metrics[book].append(r['coverage']['gt_coverage'])
    
    books = list(book_metrics.keys())
    means = [np.mean(book_metrics[b]) for b in books]
    axes[1, 1].barh(books, means, color='steelblue')
    axes[1, 1].set_xlabel('Mean GT Coverage')
    axes[1, 1].set_title('Coverage by Book')
    axes[1, 1].set_xlim(0, 1)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'summary_plots.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Summary plots saved to {save_dir}/summary_plots.png")


## 4. Run Evaluation


In [ ]:
# Test on single image first
print("Testing on single image...")

pipeline.load_models()

# Get first sample (skip cover pages)
TEST_IDX = 5
pil_image, image_info, gt_texts = dataset[TEST_IDX]
image = np.array(pil_image)

print(f"Image: {image_info['book_title']} page {image_info['page_index']}")
print(f"GT texts: {len(gt_texts)}")

# Run evaluation
result = evaluate_page(
    pipeline, image, gt_texts, 
    containment_threshold=CONTAINMENT_THRESHOLD,
    conf_threshold=0.5
)

print(f"\nResults:")
print(f"  OCR samples: {len(result['ocr_predictions'])}")
print(f"  Trans samples: {len(result['trans_predictions'])}")
print(f"  Coverage: {result['coverage']}")
print(f"  Ordering: {result['ordering']}")

# Save test visualization
if result['pred_bboxes']:
    test_vis_path = os.path.join(RUN_DIR, "test_visualization.png")
    visualize_matching(
        image, 
        result['pred_bboxes'], 
        result['valid_gt_texts'],
        result['matches'],
        save_path=test_vis_path
    )
    print(f"\nTest visualization saved to: {test_vis_path}")


In [ ]:
# Show OCR and Translation comparisons
print("OCR Comparison:")
for i, (pred, exp) in enumerate(zip(result['ocr_predictions'][:5], result['ocr_expected'][:5])):
    print(f"  [{i}] Pred: {pred}")
    print(f"      GT:   {exp}")
    print()

print("\nTranslation Comparison:")
for i, (pred, exp) in enumerate(zip(result['trans_predictions'][:5], result['trans_expected'][:5])):
    print(f"  [{i}] Pred: {pred}")
    print(f"      GT:   {exp}")
    print()


## 5. Full Dataset Evaluation


In [ ]:
# Run full evaluation with visualization saving
all_results = []
VIS_SAMPLE_INTERVAL = 10  # Save visualization every N pages

print(f"Evaluating {len(dataset)} pages...")
print(f"Saving visualizations every {VIS_SAMPLE_INTERVAL} pages to {VIS_DIR}")

for idx in tqdm(range(len(dataset))):
    pil_image, image_info, gt_texts = dataset[idx]
    image = np.array(pil_image)
    
    result = evaluate_page(
        pipeline, image, gt_texts,
        containment_threshold=CONTAINMENT_THRESHOLD,
        conf_threshold=0.5
    )
    
    result['image_info'] = image_info
    all_results.append(result)
    
    # Save visualization for sample pages
    if idx % VIS_SAMPLE_INTERVAL == 0 and result['pred_bboxes']:
        vis_path = os.path.join(VIS_DIR, f"page_{idx:04d}_{image_info['book_title']}.png")
        visualize_matching(
            image, 
            result['pred_bboxes'], 
            result['valid_gt_texts'],
            result['matches'],
            save_path=vis_path
        )
    
    # Save translation output for every page
    trans_path = os.path.join(TRANS_DIR, f"page_{idx:04d}_{image_info['book_title']}.json")
    save_translation_output(
        result['ocr_predictions'],
        result['ocr_expected'],
        result['trans_predictions'],
        result['trans_expected'],
        result.get('trans_sources', []),
        image_info,
        trans_path
    )

print("Evaluation complete!")


In [ ]:
# Compute final metrics
metrics = compute_final_metrics(all_results, device=DEVICE)

print("=" * 60)
print("PIPELINE EVALUATION RESULTS")
print("=" * 60)

print("\n📝 OCR Metrics (Matched Bubbles):")
print(f"   CER: {metrics['ocr_cer']:.4f}")

print("\n🌐 Translation Metrics (Matched Bubbles):")
print(f"   CER:     {metrics['trans_cer']:.4f}")
print(f"   WER:     {metrics['trans_wer']:.4f}")
print(f"   BLEU:    {metrics['trans_bleu']:.4f}")
print(f"   chrF++:  {metrics['trans_chrf_pp']:.4f}")
print(f"   BERTScore F1:     {metrics['trans_bertscore_f1']:.4f}")
print(f"   BERTScore P:      {metrics['trans_bertscore_precision']:.4f}")
print(f"   BERTScore R:      {metrics['trans_bertscore_recall']:.4f}")
print(f"   COMET:            {metrics['trans_comet']:.4f}")

print("\n📖 Ordering Metrics:")
print(f"   Kendall's Tau:     {metrics['ordering_kendall_tau']:.4f}")
print(f"   Exact Match Rate:  {metrics['ordering_exact_match_rate']:.4f}")

print("\n📊 Coverage Metrics:")
print(f"   GT Coverage:        {metrics['gt_coverage_mean']:.4f}")
print(f"   Bubble Utilization: {metrics['bubble_utilization_mean']:.4f}")

print("\n📈 Counts:")
print(f"   Pages:         {metrics['num_pages']}")
print(f"   OCR Samples:   {metrics['num_ocr_samples']}")
print(f"   Trans Samples: {metrics['num_trans_samples']}")

print("=" * 60)


## 6. Save Results


In [ ]:
# Save metrics
metrics_path = os.path.join(RUN_DIR, "metrics.json")
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to: {metrics_path}")

# Save detailed results
detailed_results = []
for r in all_results:
    detailed_results.append({
        'image_info': r['image_info'],
        'coverage': r['coverage'],
        'ordering': {k: float(v) if isinstance(v, (np.floating, np.integer)) else v 
                     for k, v in r['ordering'].items()},
        'num_ocr_samples': len(r['ocr_predictions']),
        'num_trans_samples': len(r['trans_predictions'])
    })

results_path = os.path.join(RUN_DIR, "detailed_results.json")
with open(results_path, 'w') as f:
    json.dump(detailed_results, f, indent=2, default=str)
print(f"Detailed results saved to: {results_path}")

# Save all OCR and translation outputs combined
all_outputs = {
    'ocr': {'predictions': [], 'expected': []},
    'translation': {'predictions': [], 'expected': [], 'sources': []}
}
for r in all_results:
    all_outputs['ocr']['predictions'].extend(r['ocr_predictions'])
    all_outputs['ocr']['expected'].extend(r['ocr_expected'])
    all_outputs['translation']['predictions'].extend(r['trans_predictions'])
    all_outputs['translation']['expected'].extend(r['trans_expected'])
    all_outputs['translation']['sources'].extend(r.get('trans_sources', []))

outputs_path = os.path.join(RUN_DIR, "all_outputs.json")
with open(outputs_path, 'w', encoding='utf-8') as f:
    json.dump(all_outputs, f, ensure_ascii=False, indent=2)
print(f"All outputs saved to: {outputs_path}")

# Create summary plots
create_summary_plots(all_results, PLOTS_DIR)

# Save config
config = {
    'device': DEVICE,
    'containment_threshold': CONTAINMENT_THRESHOLD,
    'num_pages': len(dataset),
    'run_id': RUN_ID,
    'openmantra_root': OPENMANTRA_ROOT
}
config_path = os.path.join(RUN_DIR, "config.json")
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"Config saved to: {config_path}")

print(f"\n✅ All results saved to: {RUN_DIR}")


In [ ]:
# Cleanup
pipeline.unload_models()
print("Models unloaded.")


## 7. Analysis

### Per-Book Breakdown


In [ ]:
# Per-book analysis
from collections import defaultdict

book_results = defaultdict(list)
for r in all_results:
    book_title = r['image_info']['book_title']
    book_results[book_title].append(r)

print("Per-Book Coverage:")
print("-" * 50)
for book_title, results in book_results.items():
    coverages = [r['coverage']['gt_coverage'] for r in results]
    taus = [r['ordering']['kendall_tau'] for r in results if r['ordering']['num_matched_bubbles'] >= 2]
    
    print(f"{book_title}:")
    print(f"   Pages: {len(results)}")
    print(f"   Avg GT Coverage: {np.mean(coverages):.4f}")
    print(f"   Avg Kendall Tau: {np.mean(taus):.4f}" if taus else "   Avg Kendall Tau: N/A")
    print()


## Summary

### Metrics Interpretation

| Metric | Description | Good Value |
|--------|-------------|------------|
| **OCR CER** | Character Error Rate | Lower is better (<0.1) |
| **Trans BLEU** | Translation quality (n-gram overlap) | Higher is better (>0.3) |
| **Trans chrF++** | Translation quality (char-level) | Higher is better (>0.4) |
| **Trans BERTScore F1** | Semantic similarity (BERT embeddings) | Higher is better (>0.8) |
| **Trans COMET** | Translation quality (neural, uses source) | Higher is better (>0.5) |
| **Kendall's Tau** | Reading order correlation | Higher is better (>0.8) |
| **Exact Match Rate** | % pages with perfect order | Higher is better |
| **GT Coverage** | % of GT texts matched to bubbles | Higher is better (>0.8) |
| **Bubble Utilization** | % of bubbles containing GT text | Higher is better |

### Notes
- Metrics are computed only on **matched** bubble-text pairs
- **BERTScore** measures semantic similarity but can be forgiving of grammar/word order issues
- **COMET** uses source text context and correlates well with human judgments
- Low GT Coverage may indicate:
  - Bubble detection misses
  - Containment threshold too strict
  - Text outside bubbles in dataset
- Low Kendall's Tau may indicate ordering algorithm issues
